# M02-04 — Ingesta desde Mongo (extra)

[← Anterior](04-lab-calidad-limpieza.ipynb) · [Siguiente →](../M03-transformacion-datos/01-teoria.ipynb)

Este fichero es el **guion**. No lo rellenes aquí: **crea tu propio notebook** y ve construyéndolo celda a celda.

## Qué vas a hacer

**Extra.** No forma parte del pipeline: M03 no lo necesita. El resto del curso sigue leyendo ficheros.

Aquí Spark se conecta a un **proceso Mongo vivo** (contenedor `mongo` del compose). Lees la colección `novashop.reviews`, insertas un documento y el `count` sube. Eso no se puede fingir con un JSONL.

El Codespace tiene que estar **reconstruido** con el `docker-compose` nuevo. Un `git pull` no levanta Mongo. Si el ping falla: paleta (`F1`) → **Dev Containers: Rebuild Container**, o crea un Codespace nuevo.

## 0 — Crea tu notebook

1. En el explorador, abre la carpeta `notebooks/trabajo/`.
2. Clic derecho → **New File…**
3. Nombre exacto: `M02-04-ingesta-mongo.ipynb` (incluye `.ipynb`).
4. Ábrelo. Arriba a la derecha (o `F1` → `Notebook: Select Notebook Kernel`) elige **Python (NovaShop)**.
5. Deja **este** guion a un lado (pestaña) y escribe **solo** en el tuyo.

## Cómo organizar *tu* notebook (siempre)

En cada paso creas **dos celdas**, en este orden:

1. **Markdown** — qué vas a hacer y por qué, con tus palabras.
2. **Código** — el de la celda de código del paso. Lo ejecutas (`Shift+Enter`), miras la salida y, si no cuadra, lo mejoras.

No dejes un muro de código sin explicación. Un notebook se lee de arriba abajo, como un cuaderno.

> Kernel **Python (NovaShop)**. Si no aparece: terminal → `bash .devcontainer/setup.sh` → vuelve a elegir kernel.


### Paso 1 — Arranque con el connector

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Celda 0 y una sesión **nueva** que baja el connector de Mongo. Si el kernel ya tenía Spark (otro lab), `getOrCreate` reusa esa sesión **sin** el jar: por eso paramos antes.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** Versión `3.5.x`, master `local[*]`, URI `mongodb://mongo:27017` (en el Codespace con compose).

**Por qué este paso.** El string de `get_spark` sigue siendo solo el nombre de la app. El connector va en `packages`.

**Si no sale.** Si `format(mongodb)` falla más abajo: no paraste la sesión vieja. `spark.stop()` y esta celda otra vez.


In [ ]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


from pyspark.sql import SparkSession
from session import MONGO_SPARK_PACKAGE, mongo_uri

active = SparkSession.getActiveSession()
if active is not None:
    active.stop()

spark = get_spark("novashop-mongo", packages=MONGO_SPARK_PACKAGE)
print(spark.version, spark.sparkContext.master)
print("MONGO_URI", mongo_uri())


### Paso 2 — Ping al proceso

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Antes de Spark, compruebo que hay un servidor escuchando. `pymongo` habla con Mongo; Spark aún no.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** `{'ok': 1.0}` (o similar) y en la lista aparece `novashop` (tras el seed del setup).

**Por qué este paso.** Si esto funciona, el sistema está **vivo**. El fallo típico no es Spark: es el contenedor que no está.

**Si no sale.** Timeout / `mongo: Name or service not known`: Codespace antiguo. Rebuild o Codespace nuevo. En local sin compose, este lab no se puede hacer.


In [ ]:
from pymongo import MongoClient
from session import mongo_uri

client = MongoClient(mongo_uri(), serverSelectionTimeoutMS=8000)
print(client.admin.command("ping"))
print("databases", client.list_database_names())


### Paso 3 — Cuenta con el driver

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Mismo servidor, sin Spark. Así separas “Mongo tiene 150 docs” de “Spark los lee”.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** `pymongo count 150` y un dict con `review_id`, `stars`, `meta` (anidado).

**Por qué este paso.** 150 es el seed (`data/raw/reviews.jsonl` → Mongo en el `setup`). El JSONL es la copia de arranque; a partir de aquí el origen es la base.


In [ ]:
col = client["novashop"]["reviews"]
print("pymongo count", col.count_documents({}))
col.find_one()


### Paso 4 — Spark lee la colección

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

`format("mongodb")` no es un fichero. La URI apunta al proceso del paso 2. Cada `count`/`show` es una lectura batch (no streaming).

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** `spark count 150`. Schema con `_id` (lo pone Mongo) y `meta` struct. `show` pinta filas que salen del servidor, no de `data/raw/`.

**Por qué este paso.** Si el schema no tiene `_id`, no estás leyendo Mongo (estás leyendo el JSONL). Revisa `format` y la URI.

**Si no sale.** `Failed to find data source: mongodb`: sesión sin connector → paso 1 otra vez.


In [ ]:
from session import mongo_uri

uri = mongo_uri()
reviews = (
    spark.read.format("mongodb")
    .option("spark.mongodb.read.connection.uri", uri)
    .option("spark.mongodb.read.database", "novashop")
    .option("spark.mongodb.read.collection", "reviews")
    .load()
)
print("spark count", reviews.count())
reviews.printSchema()
reviews.show(3, truncate=False)


### Paso 5 — Documento anidado

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

En CSV no hay `meta.verified`. Aquí el schema-on-read **entra** al struct. Ocho reseñas vienen sin producto (suciedad del generador).

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** Columnas `verified` y `lang`. `product_id nulos` **8**.

**Por qué este paso.** Spark no “aplasta” el JSON: declara un struct. Es la misma idea que el schema de M02-02, pero el origen es BSON.


In [ ]:
from pyspark.sql.functions import col

reviews.select("review_id", "stars", "meta.verified", "meta.lang").show(5)
print("product_id nulos", reviews.where(col("product_id").isNull()).count())


### Paso 6 — Insertas un documento

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

Esto es lo que no puedes hacer con un CSV del repo: cambias el sistema **ahora**. `pymongo` escribe; Spark aún no se ha enterado.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** `inserted_id` es un ObjectId. `pymongo count` **151**.

**Por qué este paso.** Si ejecutas esta celda dos veces, falla o duplica: `review_id` no es único en Mongo salvo que pongas índice. Si ya insertaste, o borra `R99999` o usa otro id.


In [ ]:
from datetime import datetime, timezone

col = client["novashop"]["reviews"]
doc = {
    "review_id": "R99999",
    "customer_id": "C0001",
    "product_id": "P001",
    "stars": 5,
    "comment": "insertado en el lab",
    "created_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S"),
    "channel": "web",
    "meta": {"verified": True, "lang": "es", "source": "lab"},
}
res = col.insert_one(doc)
print("inserted_id", res.inserted_id)
print("pymongo count", col.count_documents({}))


### Paso 7 — Spark vuelve a leer: +1

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

No reutilices el DataFrame viejo “de memoria”: vuelves a `load()`. Spark no se queda escuchando; cada lectura es una foto nueva.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** `spark count` **151** y una fila `insertado en el lab` / `source=lab`.

**Por qué este paso.** Si sigue 150: o no re-leíste, o insertaste en otra base/colección. Mira `mongo_uri()` y `novashop.reviews`.


In [ ]:
from session import mongo_uri

uri = mongo_uri()
reviews = (
    spark.read.format("mongodb")
    .option("spark.mongodb.read.connection.uri", uri)
    .option("spark.mongodb.read.database", "novashop")
    .option("spark.mongodb.read.collection", "reviews")
    .load()
)
from pyspark.sql.functions import col

print("spark count", reviews.count())
reviews.where(col("review_id") == "R99999").select(
    "review_id", "comment", "meta.source"
).show(truncate=False)


### Paso 8 — Copia a staging (Parquet)

En *tu* notebook: una celda Markdown que explique esto (con tus palabras):

A partir de aquí el curso vuelve a ficheros. Escribes una foto de la colección; M03 no la usa.

Debajo, una celda de código. El código está **en la celda siguiente** (márcalo y llévatelo). Ejecuta (`Shift+Enter`).

**Comprueba.** `parquet 151` (o 150 si saltaste el insert). Carpeta `data/staging/reviews_raw/`.

**Por qué este paso.** El live queda en Mongo; el pipeline del curso no depende de que Mongo siga encendido.


In [ ]:
from paths import ensure_dirs

ensure_dirs()
dest = STAGING / "reviews_raw"
reviews.write.mode("overwrite").parquet(str(dest))
print("parquet", spark.read.parquet(str(dest)).count())


## Comprueba

Antes de dar el lab por cerrado, vuelve a ejecutar de arriba abajo (**Run All**) y verifica:

`ping` ok. Spark `format("mongodb")` cuenta **150** al seed y **151** tras el insert de `R99999`.
Parquet en `STAGING/reviews_raw`. M03 no pide este fichero.


## Mejora — Solo verificadas con 4+ estrellas

Sobre la lectura Spark, filtra `meta.verified` y `stars >= 4`, cuenta y muestra 5. Markdown: cuántas hay y por qué un filtro sobre struct no es un join.

Si te atasca, el código está en la celda siguiente.


In [ ]:
buenas = reviews.where(col("meta.verified") & (col("stars") >= 4))
print(buenas.count())
buenas.select("review_id", "stars", "meta.lang").show(5)


## Si algo falla

| Qué ves | Suele ser | Qué haces |
|---------|-----------|-----------|
| Timeout / no host `mongo` | Compose no está (Codespace viejo o local sin Docker) | Rebuild Container o Codespace nuevo. `git pull` no basta |
| Failed to find data source: mongodb | Sesión Spark sin el connector | `spark.stop()` y el paso 1 (`packages=MONGO_SPARK_PACKAGE`) |
| count Spark sigue 150 | No volviste a `load()` o insertaste en otro sitio | Paso 7 entero; comprueba `novashop.reviews` con pymongo |
| Duplicate key / 152+ | Re-ejecutaste el insert | Usa otro `review_id` o `col.delete_one({"review_id": "R99999"})` |


## Siguiente

Cuando hayas **comprobado** y (si quieres) **mejorado**, abre [M03 — teoría (el extra acaba aquí)](../M03-transformacion-datos/01-teoria.ipynb).
